# Modelado Predictivo: Clasificación con Regresion Logistica
**Proyecto:** Clasificador de Salud Fetal (Dataset Cardiotocography - UCI)

**Contexto:** Este modelado busca la clasificiacion de 2 clases (1 normal y 3 patologico) en el dataset, usando estas en el entrenamiento y sumando la clasificacion de 

### Objetivos del Modelado

## Importacion de librerias escenciales y 

In [42]:
# Importación de librerías esenciales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Modificación para importar los modelos fuera de la carpeta actual
import os
import sys

root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)

# Importación del modelo
from src.LogisticRegression import LogisticRegression
from src.utils_and_metrics import train_test_split, accuracy, confusion_matrix, recall, f1_score, precision

El 

In [34]:
#Cargar el dataset
df = pd.read_csv('../data/processed/CTG_clean.csv')
df.head()

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,NSP
0,120.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006380,0.0,0.006380,0.003190,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003322,0.0,0.008306,0.003322,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.002561,0.0,0.007682,0.002561,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.006515,0.0,0.008143,0.000000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [35]:


"""""
#separacion para clasificar unicamente clase 1 y clase 3, descartando la clase 2

#Descartar las filas de la clase 2
df_filtrado = df[df['NSP'] != 2].copy()

#Separar y
X_raw = df_filtrado.drop(columns=['NSP']).to_numpy()

#Convertir NSP: Clase 1 = 0 (Normal) y Clase = 1 (Patologico)
y = np.where(df_filtrado['NSP'].to_numpy() == 3, 1, 0)

#Estandarización
mean = X_raw.mean(axis=0)
std = X_raw.std(axis=0)
std = np.where(std == 0, 1.0, std) # Evita división por cero en caso de características constantes
X = (X_raw - mean) / std

"""""

'""\n#separacion para clasificar unicamente clase 1 y clase 3, descartando la clase 2\n\n#Descartar las filas de la clase 2\ndf_filtrado = df[df[\'NSP\'] != 2].copy()\n\n#Separar y\nX_raw = df_filtrado.drop(columns=[\'NSP\']).to_numpy()\n\n#Convertir NSP: Clase 1 = 0 (Normal) y Clase = 1 (Patologico)\ny = np.where(df_filtrado[\'NSP\'].to_numpy() == 3, 1, 0)\n\n#Estandarización\nmean = X_raw.mean(axis=0)\nstd = X_raw.std(axis=0)\nstd = np.where(std == 0, 1.0, std) # Evita división por cero en caso de características constantes\nX = (X_raw - mean) / std\n\n'

In [36]:

# separar etiquetas
X_todo = df.drop(columns=['NSP']).to_numpy()
y_todo_og = df['NSP'].to_numpy()

# partir antes de filtrar
def train_test_split(X, y, test_size=0.25, seed=0):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(X))
    idx_train_end = int((1 - test_size) * len(X))
    train_idx = indices[:idx_train_end]
    test_idx = indices[idx_train_end:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

#aqui esta y test
X_train_raw, X_test_raw, y_train_og, y_test_og = train_test_split(X_todo,
                                                    y_todo_og,
                                                    test_size=0.20,
                                                    seed=0)#,seed=0

#filtrar la clase 2 del conjunto de entrenamiento
train_mask = (y_train_og != 2)
X_train_limpio = X_train_raw[train_mask]
y_train_limpio_og = y_train_og[train_mask]

#convertir clase 1 y 3 a binario 1 = 0 y 3 = 1
y_train = np.where(y_train_limpio_og == 3, 1, 0)

#std solo con X_train_limpio
mean = X_train_limpio.mean(axis=0)
std = X_train_limpio.std(axis=0)
std = np.where(std == 0, 1.0, std)

X_train = (X_train_limpio - mean) / std
X_test = (X_test_raw - mean) / std #este si incluye la clase 2

In [37]:

#Visualización e impresión de dimensiones
display(df.head())
print(f"Dimensión de Matriz X (Características): {X_todo.shape}")
print(f"Dimensión de Vector y (Objetivo NSP): {y_todo_og.shape}")
print(f"Entrenamiento: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Prueba:        X_test={X_test.shape}, y_test={y_test_og.shape}")
print(f"Clases en y:   {np.unique(y_train)}")

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,NSP
0,120.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006380,0.0,0.006380,0.003190,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003322,0.0,0.008306,0.003322,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.002561,0.0,0.007682,0.002561,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.006515,0.0,0.008143,0.000000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


Dimensión de Matriz X (Características): (2126, 21)
Dimensión de Vector y (Objetivo NSP): (2126,)
Entrenamiento: X_train=(1460, 21), y_train=(1460,)
Prueba:        X_test=(426, 21), y_test=(426,)
Clases en y:   [0 1]


### Resultados de Entrenamiento
Elementos en clase 0 o clase 1

In [ ]:
# Entrenar el modelo de Regresión Logística
model = LogisticRegression(learning_rate = 0.1, n_iterations = 2000)
model.fit(X_train, y_train)

#predicciones_train = model.predict(X_train)
#no incluye la clase 2 en el entrenamiento, pero si en la prueba
prediction_entrenamiento = model.predict(X_train)
print("Distribución de clases predichas (0 = Normal, 1 = Patológico):", np.bincount(prediction_entrenamiento))

#Predicciones
prediction_prueba = model.predict(X_test)
print("Distribución de clases predichas (0 = Normal, 1 = Patológico):", np.bincount(prediction_prueba))
predicciones_prueba_og = np.where(prediction_prueba == 1, 3, 1)
print("Distribución de clases predichas (1 = Normal, 3 = Patológico):", np.bincount(predicciones_prueba_og))

# Metricas de entrenamiento impresas
print("Accuracy:", accuracy(y_train, prediction_entrenamiento))
print("Matriz de confusión:\n", confusion_matrix(y_train, prediction_entrenamiento))
print("Precisión por clase:", precision(confusion_matrix(y_train, prediction_entrenamiento)))
print("Sensibilidad (recall) por clase:", recall(confusion_matrix(y_train, prediction_entrenamiento)))
print("F1-Score por clase:", f1_score(confusion_matrix(y_train, prediction_entrenamiento)))


Distribución de clases predichas (0 = Normal, 1 = Patológico): [1305  155]
Distribución de clases predichas (0 = Normal, 1 = Patológico): [364  62]
Distribución de clases predichas (1 = Normal, 3 = Patológico): [  0 364   0  62]
Accuracy: 0.9835616438356164
Matriz de confusión:
 [[1299   18]
 [   6  137]]
Precisión por clase: [0.9954023  0.88387097]
Sensibilidad (recall) por clase: [0.98633257 0.95804196]
F1-Score por clase: [0.99084668 0.91946309]


### Resultados de test

Resultados incluyendo la clase 2 en la prediccion para entrar en la clasifciacion de clase 1 o clase 3

De 338 muestras reales de Clase 1 en X_test -> Asignadas a Clase 1: 330 (TP) | Asignadas a Clase 3: 8 (FP)

De 55 muestras reales de Clase 2 en X_test -> Asignadas a Clase 1: 34 | Asignadas a Clase 3: 21

De 33 muestras reales de Clase 3 en X_test -> Asignadas a Clase 1: 0(FN) | Asignadas a Clase 3: 33 (TN)

In [ ]:
#predicciones de test
predicciones = model.predict(X_test)

#imprimir métricas de evaluación
print("Accuracy:", accuracy(y_test_og, predicciones_prueba_og))
print("Matriz de confusión:\n", confusion_matrix(y_test_og, predicciones_prueba_og))
print("Precisión por clase:", precision(confusion_matrix(y_test_og, predicciones_prueba_og)))
print("Sensibilidad (recall) por clase:", recall(confusion_matrix(y_test_og, predicciones_prueba_og)))
print("F1-Score por clase:", f1_score(confusion_matrix(y_test_og, predicciones_prueba_og)))

Accuracy: 0.018779342723004695
Matriz de confusión:
 [[  0   0   0   0]
 [  0 330   0   8]
 [  0  34   0  21]
 [  0   0   0  33]]
Precisión por clase: [0.         0.90659341 0.         0.53225806]
Sensibilidad (recall) por clase: [0.         0.97633136 0.         1.        ]
F1-Score por clase: [0.         0.94017094 0.         0.69473684]


In [45]:
#

Añadir curva roc y grica de regresion logistica

# **Conclusiones**
